In [1]:
!pip install evaluate
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 41.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 19.0.1
    Uninstalling pyarrow-19.0.1:
      Successfully uninstalled pyarrow-19.0.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pylibcudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
cudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
bigframes 2.12.0 requires google-cloud-bigquery[bqstorage,pandas]>=3.31.0, but you have google-cloud-bigquery 3.25.0 which is incompatible.


In [2]:
import kagglehub
import os
import torch
import evaluate
import polars as pl
import pandas as pd
import numpy as np

from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

2025-11-02 15:29:05.991522: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762097346.161064      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762097346.211506      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
path = "/kaggle/input/mrbeast-youtube-comment-sentiment-analysis/sentiment_analysis_dataset.csv"

df = pd.read_csv(path, on_bad_lines='skip')
df = pl.from_pandas(df)
df = df.drop_nulls()

In [4]:
mapping = {"Neutral" : 0, "Negative" : 1, "Positive" : 2, " Neutral" : 0}

df = df.with_columns(
    pl.col('Sentiment').replace(mapping).cast(pl.Int64)
).rename({'Sentiment' : 'label'})

In [5]:
test_size = 25
train_df = df[0:len(df)-test_size]
test_df = df[len(df)-test_size:].to_dicts()

In [6]:
dataset = Dataset.from_list(train_df.to_dicts())
dataset = dataset.train_test_split(test_size=0.2, shuffle=True)

In [7]:
checkpoint = "google-bert/bert-base-uncased"

id2label = {0 : "Neutral", 1 : "Negative", 2 : "Positive"}
label2id = {v : k for k, v, in id2label.items()}

tokenizer = AutoTokenizer.from_pretrained(
    checkpoint
)

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels=3, id2label=id2label, label2id=label2id, trust_remote_code=True
)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
def tokenize(example):
    tokenized = tokenizer(example['Comment'], truncation=True)
    return tokenized

dataset = dataset.map(tokenize, batched=True, remove_columns=['Comment'])
dataset

Map:   0%|          | 0/4905 [00:00<?, ? examples/s]

Map:   0%|          | 0/1227 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 4905
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1227
    })
})

In [9]:
accuracy = evaluate.load("accuracy")

def compute_metrics(example):
    preds, labels = example
    preds = np.argmax(preds, axis=1)
    return accuracy.compute(predictions=preds, references=labels)

In [10]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

training_args = TrainingArguments(
    report_to="none",
    output_dir="results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.125557,0.956805
2,No log,0.037445,0.991035
3,No log,0.042121,0.992665
4,0.142000,0.050934,0.990220
5,0.142000,0.049338,0.991035


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=770, training_loss=0.09342835816470059, metrics={'train_runtime': 190.5075, 'train_samples_per_second': 128.735, 'train_steps_per_second': 4.042, 'total_flos': 446549206396482.0, 'train_loss': 0.09342835816470059, 'epoch': 5.0})

In [11]:
def inference(sample, model, tokenizer, device):
    inputs = tokenizer(sample['Comment'], return_tensors="pt").to(device)
    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)

    outputs = outputs.logits.argmax().item()
    label = model.config.id2label

    return f"Input text : {sample['Comment']}, Actual Label : {label[sample['label']]}, Predicted Label : {label[int(outputs)]}\n"

In [12]:
eval_model = AutoModelForSequenceClassification.from_pretrained(
    '/kaggle/working/results/checkpoint-770'
)

eval_tokenizer = AutoTokenizer.from_pretrained(
    '/kaggle/working/results/checkpoint-770'
)

for sample in test_df:
    current_ouput = inference(
        sample=sample,
        model=eval_model,
        tokenizer=eval_tokenizer,
        device="cpu"
    )
    print(current_ouput)

Input text : This is the best part ever!!!!!!!!, Actual Label : Positive, Predicted Label : Positive

Input text : is that him..?, Actual Label : Neutral, Predicted Label : Neutral

Input text : WOW nice video, Actual Label : Positive, Predicted Label : Positive

Input text : at this moment he realised, Actual Label : Neutral, Predicted Label : Neutral

Input text : roUnD, Actual Label : Neutral, Predicted Label : Neutral

Input text : Thanku for . subscriber, Actual Label : Positive, Predicted Label : Positive

Input text : Your going to what, Actual Label : Neutral, Predicted Label : Neutral

Input text : alright buddy, Actual Label : Neutral, Predicted Label : Positive

Input text : No way!!!!!!!, Actual Label : Positive, Predicted Label : Positive

Input text : flaxan mark, Actual Label : Neutral, Predicted Label : Neutral

Input text : LIFE IS ROBLO?, Actual Label : Neutral, Predicted Label : Neutral

Input text : Was Finny, Actual Label : Positive, Predicted Label : Neutral

Inpu